<a href="https://colab.research.google.com/github/corrielynnyuill-debug/AIProject_Stocks-CLY/blob/main/AIProject_Stocks_Part3_CLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount drive in Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets
# Set file path
DATA_PATH = "/content/drive/MyDrive/AIProject/clean_data/"

# Load to DF
df = pd.read_parquet(DATA_PATH + "cleaned_stock_data.parquet")
print(df.head())
print('-'*80)
print(df.columns)


Mounted at /content/drive


In [2]:
# Feature engineering and technical indicators
# Apply EMA (Exponential Moving Average)
def ema(series, span):
  return series.ewm(span=span, adjust=False).mean()

# MACD (Moving Average Convergence Divergence) implementation
df = df.sort_values(['ticker', 'date'])

# MACD components
df['ema12'] = df.groupby('ticker')['close'].transform(lambda x: ema(x, span=12))
df['ema26'] = df.groupby('ticker')['close'].transform(lambda x: ema(x, span=26))

df['macd'] = df['ema12'] - df['ema26']
df['macd_signal'] = df.groupby('ticker')['macd'].transform(lambda x: ema(x, span=9))
df['macd_hist'] = df['macd'] - df['macd_signal']

# MACD crossovers
df['macd_prev'] = df.groupby('ticker')['macd'].shift(1)
df['macd_signal_prev'] = df.groupby('ticker')['macd_signal'].shift(1)

# Buy when MACD crosses above signal
df['macd_buy'] = (
    (df['macd_prev'] < df['macd_signal_prev']) &
    (df['macd'] > df['macd_signal'])
)

# Sell when MACD crosses below signal
df['macd_sell'] = (
    (df['macd_prev'] > df['macd_signal_prev']) &
    (df['macd'] < df['macd_signal'])
)

# RSI (Relative Strength Index) implementation with EMA
window = 14

# Price changes per ticker
delta = df.groupby('ticker')['close'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# EMA of gains and losses
avg_gain = gain.groupby(df['ticker']).transform(
    lambda x: x.ewm(alpha=1/window, adjust=False).mean())
avg_loss = loss.groupby(df['ticker']).transform(
    lambda x: x.ewm(alpha=1/window, adjust=False).mean())

RS = avg_gain / avg_loss
df['RSI'] = 100 - (100 / (1 + RS))

# RSI buy/sell signals
df['RSI_buy'] = df['RSI'] < 30
df['RSI_sell'] = df['RSI'] > 70

# Combine MACD and RSI
df['signal'] = 0 # default hold

df.loc[df['macd_buy'] & df['RSI_buy'], 'signal'] = 1 # Buy
df.loc[df['macd_sell'] & df['RSI_sell'], 'signal'] = -1 # Sell

print(df['signal'].value_counts())


signal
 0    20951021
-1       12693
 1       10175
Name: count, dtype: int64


In [10]:
# Prepare dataset for ML

# reduce dataset to one ticker to avoid RAM crash
ticker = df['ticker'].unique()[0]
df_small = df[df['ticker'] == ticker].copy()

features = [
    'close', 'volume',
    'rolling_7', 'rolling_30', 'volatility',
    'macd', 'macd_signal', 'macd_hist',
    'RSI'
]

df_small = df_small.dropna(subset=features + ['signal']).copy()

# Create X and y
X = df_small[features]
y = df_small['signal']

# Train test split
split_idx = int(len(df_small) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print('-'*80)



--------------------------------------------------------------------------------


In [8]:
df_small[features].isna().sum()

,0
close,0
volume,0
rolling_7,6
rolling_30,29
volatility,6
macd,0
macd_signal,0
macd_hist,0
RSI,1


In [16]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    multi_class='multinomial',
    max_iter=500,
    n_jobs=1
)

log_reg.fit(X_train, y_train)

print('-'*80)


--------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [12]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    n_jobs=1,
    random_state=42
)

rf_clf.fit(X_train, y_train)

print('-'*80)


--------------------------------------------------------------------------------


In [13]:
# Support Vector Machine (SVM)
from sklearn.svm import SVC

svm = SVC(
    kernel='linear',
    probability=True
)

svm.fit(X_train, y_train)

print('-'*80)


--------------------------------------------------------------------------------


In [15]:
# Model evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Logistic Regression': log_reg,
    'Random Forest Classifier': rf_clf,
    'Support Vector Machine': svm
}

for model_name, model in models.items():
  print(f"Model: {model_name}")
  y_pred = model.predict(X_test)
  print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
  print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
  print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
  print('-'*80)

Model: Logistic Regression


ValueError: Found array with 0 sample(s) (shape=(0, 9)) while a minimum of 1 is required by LogisticRegression.